# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their field IDs. All references use the `@id` identifier.

In [ ]:
# Get available RecordSets
record_sets = metadata.recordSet
print("Available RecordSets (@id):")
for rs in record_sets:
    print(f"  - {rs['@id']}")

# Display fields for each RecordSet
for rs in record_sets:
    print(f"\nRecordSet '@id': {rs['@id']} (Name: {rs.get('name', '[no name]')})")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields and @id:")
    for f in fields:
        print(f"    - {f['@id']} (Name: {f.get('name', '[no name]')}, DataType: {f.get('dataType', '[not specified]')})")

# Show a sample of records from each RecordSet
for rs in record_sets:
    print(f"\nSample records for RecordSet {rs['@id']}:")
    records = list(dataset.records(record_set=rs['@id']))
    for i, rec in enumerate(records[:2]):  # show first 2 records
        print(f"  Record {i+1}: {rec}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each RecordSet using their @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Extracted DataFrame for RecordSet {record_set_id} (Shape: {dataframes[record_set_id].shape})")
    print(f"Columns: {dataframes[record_set_id].columns.tolist()}")

# Preview the first record set's DataFrame
main_record_set_id = record_set_ids[0]
print(f"\nPreview of DataFrame for '{main_record_set_id}':")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps such as filtering, normalizing, and grouping. Reference columns and fields using their `@id` identifiers.

In [ ]:
# Select a numeric field for analysis from the first RecordSet.
# (Replace below '@id' with actual numeric field @id found in step 2.)
main_rs_df = dataframes[main_record_set_id]

# Example: Let's suppose there's an 'age' field with @id 'https://api.app.sen.science/frontiers/7862866/age'.
# If not present, replace with another actual numeric field @id!
numeric_field_id = 'https://api.app.sen.science/frontiers/7862866/age'  # Replace if dataset field differs
group_field_id = 'https://api.app.sen.science/frontiers/7862866/sex'    # Example grouping field @id

# Confirm presence of numeric_field_id
if numeric_field_id in main_rs_df:
    threshold = 50
    filtered_df = main_rs_df[main_rs_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by group_field_id if present
    if group_field_id in filtered_df:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped statistics for {numeric_field_id} by {group_field_id}:")
        display(grouped_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Field '{numeric_field_id}' not found in DataFrame columns: {main_rs_df.columns.tolist()}")

## 5. Visualization
Visualize distributions and relationships within the dataset. Uses columns referenced by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot age distribution if present
if numeric_field_id in main_rs_df:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_rs_df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of field {numeric_field_id}")
    plt.xlabel("Age")
    plt.ylabel("Count")
    plt.show()

# Relationship between age and sex if both fields present
if numeric_field_id in main_rs_df and group_field_id in main_rs_df:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=main_rs_df[group_field_id], y=main_rs_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel("Age")
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the FAIR² dataset using its Croissant schema URL and explored its record sets and fields using their `@id` identifiers.
- Extracted data into pandas DataFrames for flexible analysis.
- Applied basic EDA operations: filtered and normalized numeric fields, grouped by categorical fields, and visualized distributions.

This approach demonstrates reproducible FAIR dataset exploration with Croissant and `mlcroissant`, leveraging semantic IDs for clarity and consistency.